# Spark Exercises 05 — Joins & Shuffles

Joins are the heart of every Foundry pipeline — and in Spark a join is also where the most expensive thing in distributed computing happens: a **shuffle**, where data flies across the network so matching keys end up on the same machine. This mission chapter covers every join type (including the SQL-flavoured `semi` / `anti`), how to read a join in `explain()`, and the **broadcast join** trick that avoids the shuffle for small tables.

**Data files:** `orders.csv`, `customers.csv`, `products.csv`, `regions.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read all four files (`header=True`, `inferSchema=True`): `orders`, `customers`, `products`, `regions`.

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
regions = spark.read.csv("data/regions.csv", header=True, inferSchema=True)
print(orders.count(), customers.count(), products.count(), regions.count())

8000 500 20 5


### 3. **Inner join** `orders` to `customers` on `customer_id` (pass the key as a string so the column is not duplicated). Show `order_id`, `customer_name`, `segment`, `quantity` for 5 rows.

In [3]:
enriched = orders.join(customers, "customer_id", "inner")
enriched.select("order_id", "customer_name", "segment", "quantity").show(5)

+----------+-----------------+--------+--------+
|  order_id|    customer_name| segment|quantity|
+----------+-----------------+--------+--------+
|ORD-000001|  Voonix Holdings|Consumer|      10|
|ORD-000002|     Mybuzz Group|Consumer|      22|
|ORD-000003|     Gigabyte LLC|     SMB|       9|
|ORD-000004| Dabtype Holdings|Consumer|      21|
|ORD-000005|Gigabyte Partners|Consumer|      23|
+----------+-----------------+--------+--------+
only showing top 5 rows



### 4. Confirm the inner join kept every order: compare `orders.join(customers, "customer_id").count()` to `orders.count()`.

In [4]:
print("joined:", orders.join(customers, "customer_id").count())
print("orders:", orders.count())

joined: 8000
orders: 8000


### 5. **Left anti join** — find customers who have **no orders at all**. (`customers.join(orders, "customer_id", "left_anti")`). How many are there, and show 5.

In [5]:
no_orders = customers.join(orders, "customer_id", "left_anti")
print("customers with no orders:", no_orders.count())
no_orders.select("customer_id", "customer_name", "segment").show(5)

customers with no orders: 40


+-----------+----------------+----------+
|customer_id|   customer_name|   segment|
+-----------+----------------+----------+
|  CUST-0461|     Dabtype Inc|  Consumer|
|  CUST-0462| Voonix Partners|  Consumer|
|  CUST-0463|Cloudkit Systems|  Consumer|
|  CUST-0464|       Zooxo Inc|  Consumer|
|  CUST-0465|      Rhybox LLC|Enterprise|
+-----------+----------------+----------+
only showing top 5 rows



### 6. **Left semi join** — keep only the customers who **do** have orders. Note a semi join returns **only the left table's columns** (it is a filter, not a real join). How many?

In [6]:
with_orders = customers.join(orders, "customer_id", "left_semi")
print("customers with orders:", with_orders.count())
with_orders.show(5)

customers with orders: 460


+-----------+-----------------+--------------------+-------+-----------+---------+----------+------------+-----------+
|customer_id|    customer_name|               email|country|       city|region_id|   segment|loyalty_tier|signup_date|
+-----------+-----------------+--------------------+-------+-----------+---------+----------+------------+-----------+
|  CUST-0001|    Skiba Systems|contact1@skiba.ex...|     SG|  Singapore|     APAC|       SMB|      Bronze| 2023-10-20|
|  CUST-0002| Dabfeed Holdings|contact2@dabfeed....|     UK|     London|       EU|Enterprise|      Silver| 2021-02-14|
|  CUST-0003| Brightlane Group|contact3@brightla...|     MX|Mexico City|    LATAM|Enterprise|      Bronze| 2021-07-12|
|  CUST-0004|         Vinte Co|contact4@vinte.ex...|     JP|      Tokyo|     APAC|  Consumer|      Bronze| 2021-04-18|
|  CUST-0005|Bluezoom Holdings|contact5@bluezoom...|     US|    Chicago|       NA|  Consumer|      Silver| 2022-09-21|
+-----------+-----------------+-----------------

### 7. **Multi-table join.** Enrich `orders` with `customers`, then `regions` (on `region_id`), then `products` (on `product_sku`). Add `revenue = quantity * unit_price` and show `order_id, region_name, category, segment, revenue` for 5 rows.

In [7]:
full = (orders
        .join(customers, "customer_id")
        .join(regions, "region_id")
        .join(products, "product_sku")
        .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2)))
full.select("order_id", "region_name", "category", "segment", "revenue").show(5)

+----------+-------------+----------+--------+-------+
|  order_id|  region_name|  category| segment|revenue|
+----------+-------------+----------+--------+-------+
|ORD-000001| Asia Pacific|     books|Consumer|  325.8|
|ORD-000002|North America|stationery|Consumer|  31.46|
|ORD-000003|       Europe|stationery|     SMB|  10.26|
|ORD-000004|North America|      home|Consumer| 149.52|
|ORD-000005|       Europe|      home|Consumer| 187.91|
+----------+-------------+----------+--------+-------+
only showing top 5 rows



### 8. Using the `full` table, compute **revenue per `region_name`** (positive quantities only), highest first.

In [8]:
full.filter(F.col("quantity") > 0) \
    .groupBy("region_name") \
    .agg(F.round(F.sum("revenue"), 2).alias("revenue")) \
    .orderBy(F.col("revenue").desc()).show()

+--------------------+---------+
|         region_name|  revenue|
+--------------------+---------+
|       North America|731133.53|
|              Europe|433862.83|
|        Asia Pacific| 411564.1|
|       Latin America|177438.07|
|Middle East & Africa| 88854.77|
+--------------------+---------+



### 9. **See the shuffle.** Call `.explain()` on `orders.join(customers, "customer_id")` and find the `Exchange` (that is the shuffle) and the `SortMergeJoin`.

In [9]:
orders.join(customers, "customer_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#18, order_id#17, order_ts#19, product_sku#20, quantity#21, unit_price#22, channel#23, payment_method#24, status#25, discount_code#26, customer_name#55, email#56, country#57, city#58, region_id#59, segment#60, loyalty_tier#61, signup_date#62]
   +- BroadcastHashJoin [customer_id#18], [customer_id#54], Inner, BuildRight, false
      :- Filter isnotnull(customer_id#18)
      :  +- FileScan csv [order_id#17,customer_id#18,order_ts#19,product_sku#20,quantity#21,unit_price#22,channel#23,payment_method#24,status#25,discount_code#26] Batched: false, DataFilters: [isnotnull(customer_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/tomerkadison/Desktop/hafifot_osint/spark_exercises/05_Join..., PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:string,customer_id:string,order_ts:timestamp,product_sku:string,quantity:int,unit...
      +- BroadcastExchange Ha

### 10. **Broadcast join.** `regions` is tiny (5 rows), so broadcasting it avoids the shuffle entirely. Wrap it in `F.broadcast(...)`, join, and call `.explain()` — you should now see `BroadcastHashJoin` instead of `SortMergeJoin`.

In [10]:
customers.join(F.broadcast(regions), "region_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [region_id#59, customer_id#54, customer_name#55, email#56, country#57, city#58, segment#60, loyalty_tier#61, signup_date#62, region_name#119, currency#120, tax_rate#121]
   +- BroadcastHashJoin [region_id#59], [region_id#118], Inner, BuildRight, false
      :- Filter isnotnull(region_id#59)
      :  +- FileScan csv [customer_id#54,customer_name#55,email#56,country#57,city#58,region_id#59,segment#60,loyalty_tier#61,signup_date#62] Batched: false, DataFilters: [isnotnull(region_id#59)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/tomerkadison/Desktop/hafifot_osint/spark_exercises/05_Join..., PartitionFilters: [], PushedFilters: [IsNotNull(region_id)], ReadSchema: struct<customer_id:string,customer_name:string,email:string,country:string,city:string,region_id:...
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=1443]
         +- Filter isnotnull(region_

### 11. **Watch out for ambiguous columns.** Both `orders` and a renamed `customers` share `customer_id`. Join on an explicit condition `orders.customer_id == customers.customer_id` and then try to select `"customer_id"`... it is ambiguous. The fix: join on the **string key** (as we did), or `drop` one side. Here, do it the safe way and select the customer's `region_id`.

In [11]:
# safe: string key -> single customer_id column, no ambiguity
orders.join(customers, "customer_id").select("order_id", "customer_id", "region_id").show(5)

+----------+-----------+---------+
|  order_id|customer_id|region_id|
+----------+-----------+---------+
|ORD-000001|  CUST-0092|     APAC|
|ORD-000002|  CUST-0016|       NA|
|ORD-000003|  CUST-0289|       EU|
|ORD-000004|  CUST-0315|       NA|
|ORD-000005|  CUST-0356|       EU|
+----------+-----------+---------+
only showing top 5 rows



### 12. Per `segment`, count **distinct customers who placed orders** and total revenue. Join orders+customers, then `groupBy('segment')` with `countDistinct('customer_id')` and `sum(quantity*unit_price)`.

In [12]:
orders.join(customers, "customer_id") \
      .filter(F.col("quantity") > 0) \
      .groupBy("segment") \
      .agg(F.countDistinct("customer_id").alias("buyers"),
           F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("revenue")) \
      .orderBy(F.col("revenue").desc()).show()

+----------+------+----------+
|   segment|buyers|   revenue|
+----------+------+----------+
|  Consumer|   254|1043946.29|
|       SMB|   128| 516710.58|
|Enterprise|    78| 282196.43|
+----------+------+----------+

